## Loading Insurance Question Answering Dataset

In [1]:
!pip install datasets


[notice] A new release of pip is available: 23.3.2 -> 24.0
[notice] To update, run: C:\Users\diya.sharma\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!pip install transformers

^C



[notice] A new release of pip is available: 23.3.2 -> 24.0
[notice] To update, run: C:\Users\diya.sharma\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
pip install accelerate


[notice] A new release of pip is available: 23.0.1 -> 24.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!pip install evaluate

     ---------------------------------------- 0.0/84.1 kB ? eta -:--:--
     -------------- ------------------------- 30.7/84.1 kB ? eta -:--:--
     -------------------------------------- 84.1/84.1 kB 947.4 kB/s eta 0:00:00



[notice] A new release of pip is available: 23.0.1 -> 24.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from datasets import load_dataset
df=load_dataset("diya22/Insurance")

Generating train split:   0%|          | 0/8 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2 [00:00<?, ? examples/s]

### Removing id column

In [2]:
df = df.remove_columns(['id'])

### Dataset

In [3]:
df

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 8
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 2
    })
})

In [4]:
import torch
from transformers import Trainer, TrainingArguments

## Loading T5 Small Model

In [5]:
from transformers import T5TokenizerFast, T5ForQuestionAnswering

tokenizer = T5TokenizerFast.from_pretrained("google-t5/t5-small")
model = T5ForQuestionAnswering.from_pretrained("google-t5/t5-small")

Some weights of T5ForQuestionAnswering were not initialized from the model checkpoint at google-t5/t5-small and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Zero Short

In [6]:
from transformers import pipeline

# Load the question answering pipeline
# generator = pipeline("question-answering")
generator = pipeline("question-answering", model=model,tokenizer=tokenizer)
# Example question and context
question2 = "Is snow damage covered by insurance?"
context2 = "The answer is most often yes, if your home has a sloped roof, and your particular policy provides for that coverage. If your roof is flat, that is probably a different story."
# Format the input as a dictionary with 'question' and 'context' keys
input_dict = {"question": question2, "context": context2}

# Perform zero-shot inference using the question answering pipeline
one_shot_answer = generator(input_dict, max_length=400, temperature=3, top_k=10)

print(f'INPUT PROMPT:\n{question2}')
print()
   
print(f'ANSWER FROM CSV:\n{context2}')
print()

print(f'MODEL GENERATION - WITH ZERO SHOT LEARNING:\n{one_shot_answer}\n')

INPUT PROMPT:
Is snow damage covered by insurance?

ANSWER FROM CSV:
The answer is most often yes, if your home has a sloped roof, and your particular policy provides for that coverage. If your roof is flat, that is probably a different story.

MODEL GENERATION - WITH ZERO SHOT LEARNING:
[{'score': 0.0011971760541200638, 'start': 130, 'end': 155, 'answer': 'is flat, that is probably'}, {'score': 0.001166097936220467, 'start': 49, 'end': 124, 'answer': 'sloped roof, and your particular policy provides for that coverage. If your'}, {'score': 0.00116287125274539, 'start': 49, 'end': 61, 'answer': 'sloped roof,'}, {'score': 0.0011331267887726426, 'start': 130, 'end': 138, 'answer': 'is flat,'}, {'score': 0.001129526412114501, 'start': 130, 'end': 157, 'answer': 'is flat, that is probably a'}, {'score': 0.001065805903635919, 'start': 49, 'end': 70, 'answer': 'sloped roof, and your'}, {'score': 0.0010591507889330387, 'start': 49, 'end': 65, 'answer': 'sloped roof, and'}, {'score': 0.00104881

## One Short

In [7]:
from transformers import pipeline

# Load the question answering pipeline
# generator = pipeline("question-answering")
generator = pipeline("question-answering", model=model,tokenizer=tokenizer)
# Example question and context
question2 = "Is snow damage covered by insurance?"
context2 = "Great question! The answer is most often yes, if your home has a sloped roof, and your particular policy provides for that coverage. If your roof is flat, that is probably a different story. Typically with a flat roof, the risk for ice or snow damage is greatly increased, so your policy is either much more expensive, or that damage is excluded. Call your agent to be certain in either case. Good luck! Thanks for asking!"

# Format the input as a dictionary with 'question' and 'context' keys
input_dict = {"question": question2, "context": context2}

# Perform zero-shot inference using the question answering pipeline
one_shot_answer = generator(input_dict, max_length=400, temperature=3, top_k=10)

print(f'INPUT PROMPT:\n{question2}')
print()
   
print(f'ANSWER FROM CSV:\n{context2}')
print()

print(f'MODEL GENERATION - WITH ZERO SHOT LEARNING:\n{one_shot_answer}\n')



 
# dialogue = "Is snow damage covered by insurance?"
# summary = "Great question! The answer is most often yes, if your home has a sloped roof, and your particular policy provides for that coverage. If your roof is flat, that is probably a different story. Typically with a flat roof, the risk for ice or snow damage is greatly increased, so your policy is either much more expensive, or that damage is excluded. Call your agent to be certain in either case. Good luck! Thanks for asking!"
 
# prompt_template = f"""On the basis of question asked :{dialogue}
#                         Generate answer :{summary} """                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                
 
# inputs = tokenizer(prompt_template, return_tensors='pt')
# output = model.generate(inputs.input_ids,max_new_tokens=10,temperature=0.1,do_sample=True)
 
# decoded_input = tokenizer.decode(           #this give decoding of the prompt
#             inputs['input_ids'][0],
#             skip_special_tokens=True)
   
# decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)   #this give decoding of the output generated by the model
 
   
   
# print()
# print(inputs)
# print()
# print(decoded_input)
# print()
 
   
# print('Example ', i + 1)
   
# print(f'INPUT PROMPT:\n{dialogue}')
# print()
   
# print(f'ANSWER FROM CSV:\n{summary}')
# print()
   
# print(f'MODEL GENERATION - WITH ZERO SHOT LEARNING:\n{decoded_output}\n')

INPUT PROMPT:
Is snow damage covered by insurance?

ANSWER FROM CSV:
Great question! The answer is most often yes, if your home has a sloped roof, and your particular policy provides for that coverage. If your roof is flat, that is probably a different story. Typically with a flat roof, the risk for ice or snow damage is greatly increased, so your policy is either much more expensive, or that damage is excluded. Call your agent to be certain in either case. Good luck! Thanks for asking!

MODEL GENERATION - WITH ZERO SHOT LEARNING:
[{'score': 0.00023465001140721142, 'start': 65, 'end': 140, 'answer': 'sloped roof, and your particular policy provides for that coverage. If your'}, {'score': 0.00023446032719220966, 'start': 146, 'end': 205, 'answer': 'is flat, that is probably a different story. Typically with'}, {'score': 0.00022868193627800792, 'start': 146, 'end': 154, 'answer': 'is flat,'}, {'score': 0.00022456664009951055, 'start': 65, 'end': 77, 'answer': 'sloped roof,'}, {'score': 0

## Few Shorts

In [8]:
from transformers import pipeline

# Load the question answering pipeline
# generator = pipeline("question-answering")
generator = pipeline("question-answering", model=model,tokenizer=tokenizer)
# Example question and context
# Define the questions and contexts
# Define the questions and contexts
questions = [
    "Is Medicare Considered Welfare?",
    "Will Life Insurance Know If I Smoke?",
    "Does Blue Cross Blue Shield Have Life Insurance?"
]

contexts = [
    "No, Medicare is not considered welfare. When most people are talking about welfare, they are talking about food stamps, Medicaid, unemployment benefits, housing assistance, child care assistance, etc. Medicare recipients have paid Social Security taxes for at least 10 years in order to receive health care coverage through Medicare.",
    "That is a great question! The answer is simple, the cost of your auto insurance for your imported car is higher because the cost to repair it, should you damage it, is more expensive than a domestically made car would cost. The expense for parts and the specialized mechanics would cost the insurance company more, so they pass that expense on to you. Thanks for asking!",
    "Blue Cross / Blue Shield is the name of the network association for a number of health insurance companies (includes Anthem and CareFirst). Many health insurance companies do also offer life insurance programs, however, it's typically not the type of coverage they specialize in and the products may therefore not be as competitive as insurance companies who feature life insurance programs as the main type of insurance they offer. Insurance companies specializing in life insurance tend to have more aggressive underwriting and can offer lower rates because they want to secure that type of business more than others. Please let me know if I can be of further assistance. Thanks very much."
]

# Perform few-shot inference for each question and context
for i in range(len(questions)):
    # Call the generator pipeline with each question and context pair
    answer = generator(question=questions[i], context=contexts[i], max_length=400, temperature=4, top_k=10)

    # Print the question and its answer
    print(f"Question: {questions[i]}")
    print()
    print('Answer From CSV:-')
    print()
    print(contexts[i])
    # print(answer)
    print('Model Generated Answer:-')
    print()
    print(f"Answer: {answer}\n")

Question: Is Medicare Considered Welfare?

Answer From CSV:-

No, Medicare is not considered welfare. When most people are talking about welfare, they are talking about food stamps, Medicaid, unemployment benefits, housing assistance, child care assistance, etc. Medicare recipients have paid Social Security taxes for at least 10 years in order to receive health care coverage through Medicare.
Model Generated Answer:-

Answer: [{'score': 0.000656425254419446, 'start': 161, 'end': 200, 'answer': 'assistance, child care assistance, etc.'}, {'score': 0.0006330749019980431, 'start': 161, 'end': 195, 'answer': 'assistance, child care assistance,'}, {'score': 0.0006289337761700153, 'start': 112, 'end': 200, 'answer': 'stamps, Medicaid, unemployment benefits, housing assistance, child care assistance, etc.'}, {'score': 0.0006065613706596196, 'start': 112, 'end': 195, 'answer': 'stamps, Medicaid, unemployment benefits, housing assistance, child care assistance,'}, {'score': 0.000567584764212369

## Tokenization Function

In [9]:
# Tokenize and encode your input and output sequences
def preprocess_function(examples):
    inputs = tokenizer(examples['question'], truncation=True,padding=True,max_length=512)
    outputs = tokenizer(examples['answer'],  truncation=True,padding=True,max_length=512)
    
    # # Update examples with inputs and outputs
    examples["input_ids"] = inputs.input_ids
    examples["attention_mask"] = inputs.attention_mask
    examples["labels"] = outputs.input_ids
    
    # start_logits = outputs.start_logits
    # end_logits = outputs.end_logits
    
    # We need to return the examples dictionary
    return examples

### Applying tokenization function to train and test dataset

In [11]:
train_dataset = df['train'].map(preprocess_function,batched=True)
test_dataset = df['test'].map(preprocess_function,batched=True)


# # train_dataset=df['train']
# train_dataset=df['train'].map(preprocess_function,batched=True)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

### Train Dataset

In [12]:
print("Shape of Train Data:- ",train_dataset.shape)
train_dataset

Shape of Train Data:-  (8, 5)


Dataset({
    features: ['question', 'answer', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 8
})

## Test Dataset

In [13]:
print("Shape of Test Data:- ",test_dataset.shape)
test_dataset

Shape of Test Data:-  (2, 5)


Dataset({
    features: ['question', 'answer', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2
})

## Removing question, answer and attention mask column

In [14]:
train_dataset = train_dataset.remove_columns(['question','answer','attention_mask'])
test_dataset = test_dataset.remove_columns(['question','answer',"attention_mask"])

In [15]:
print("Shape of Train Data:- ",train_dataset.shape)
train_dataset

Shape of Train Data:-  (8, 2)


Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 8
})

In [16]:
print("Shape of Test Data:- ",test_dataset.shape)
test_dataset

Shape of Test Data:-  (2, 2)


Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 2
})

## Giving training Arguments

In [17]:
from transformers import Trainer, TrainingArguments

output_dir = 'checkpoints'

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=3e-5,
    evaluation_strategy ="epoch",
    logging_strategy='steps',
    logging_steps=10,
    save_strategy ="epoch",
    save_steps=10,
    num_train_epochs=10,
)

## Loading metric for checking accuracy

In [21]:
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=3e-5,
    evaluation_strategy="epoch",
    # logging_strategy='steps',
    # logging_steps=10,
    # save_strategy="epoch",
    # save_steps=10,
    num_train_epochs=10,
)

## Trainer

In [22]:
from transformers import Trainer


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    
    )

## Model Training

In [23]:
trainer.train()

  0%|          | 0/20 [00:00<?, ?it/s]

ValueError: The model did not return a loss from the inputs, only the following keys: start_logits,end_logits,past_key_values,encoder_last_hidden_state. For reference, the inputs it received are input_ids.

In [22]:
trainer.evaluate()

  0%|          | 0/50 [00:00<?, ?it/s]

{'eval_loss': 0.9818434119224548,
 'eval_runtime': 35.5436,
 'eval_samples_per_second': 1.407,
 'eval_steps_per_second': 1.407,
 'epoch': 3.0}

In [ ]:
# fine_tuned_model = T5ForConditionalGeneration.from_pretrained("/content/fine-tuned-model")

In [23]:
# fine_tuned_model
 
model_path="./fine-tune-checkpoint-local1"
 
trainer.model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

('./fine-tune-checkpoint-local\\tokenizer_config.json',
 './fine-tune-checkpoint-local\\special_tokens_map.json',
 './fine-tune-checkpoint-local\\spiece.model',
 './fine-tune-checkpoint-local\\added_tokens.json')

In [31]:
from  transformers  import T5ForQuestionAnswering

model1 =T5ForQuestionAnswering.from_pretrained(model_path)
# tokenizer1 = T5Tokenizer.from_pretrained(model_path)

Some weights of T5ForQuestionAnswering were not initialized from the model checkpoint at ./fine-tune-checkpoint-local and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [33]:
from transformers import pipeline

# Load the question answering pipeline
# generator = pipeline("question-answering")
generator = pipeline("question-answering", model=model1, tokenizer=tokenizer)
# Example question and context
question2 = "Is snow damage covered by insurance?"
context2 = "Great question! The answer is most often yes, if your home has a sloped roof, and your particular policy provides for that coverage. If your roof is flat, that is probably a different story. Typically with a flat roof, the risk for ice or snow damage is greatly increased, so your policy is either much more expensive, or that damage is excluded. Call your agent to be certain in either case. Good luck! Thanks for asking!"

# Format the input as a dictionary with 'question' and 'context' keys
input_dict = {"question": question2, "context": context2}

# Perform zero-shot inference using the question answering pipeline
one_shot_answer = generator(input_dict, max_length=400, temperature=3, top_k=10)

print(one_shot_answer)

ValueError: None is not in list

In [ ]:
# model.save_pretrained("./fine-tuned-model1")

In [ ]:
# tokenizer.save_pretrained("./fine-tuned-model1")

('./fine-tuned-model1\\tokenizer_config.json',
 './fine-tuned-model1\\special_tokens_map.json',
 './fine-tuned-model1\\spiece.model',
 './fine-tuned-model1\\added_tokens.json')

In [ ]:
# example_indices = [65,50]
 
# for i, index in enumerate(example_indices):
#     dialogue = df['test'][index]['question']
#     summary = df['test'][index]['answer']
 
#     prompt_template = f"""On the basis of question asked :{dialogue}
#                         Generate the answer :{summary} """                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                
 
#     inputs = tokenizer1(prompt_template, return_tensors='pt')
#     output = model1.generate(inputs.input_ids,max_new_tokens=10,temperature=0.1,do_sample=True)
 
#     decoded_input = tokenizer1.decode(           #this give decoding of the prompt
#             inputs['input_ids'][0],
#             skip_special_tokens=True)
   
#     decoded_output = tokenizer1.decode(output[0], skip_special_tokens=True)   #this give decoding of the output generated by the model
 
   
   
#     print()
#     print(inputs)
#     print()
#     print(decoded_input)
#     print()
 
   
#     print('Example ', i + 1)
   
#     print(f'INPUT PROMPT:\n{dialogue}')
#     print()
   
#     print(f'ANSWER FROM CSV:\n{summary}')
#     print()
   
#     print(f'MODEL GENERATION - WITH ZERO SHOT LEARNING:\n{decoded_output}\n')
#     print("-------------------------------------------------------------------------------------------------------")

In [245]:
import torch
from transformers import T5ForQuestionAnswering, T5TokenizerFast
from datasets import load_dataset
from transformers import Trainer, TrainingArguments

In [294]:
# Load dataset
df = load_dataset("diya22/llm")
df = df.remove_columns(['id'])
df


DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 800
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 200
    })
})

In [247]:
# Initialize T5 model and tokenizer
tokenizer = T5TokenizerFast.from_pretrained("google-t5/t5-small")
model = T5ForQuestionAnswering.from_pretrained("google-t5/t5-small")

Some weights of T5ForQuestionAnswering were not initialized from the model checkpoint at google-t5/t5-small and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [279]:
def preprocess_function(examples):
    examples['inputs'] = tokenizer(examples['question'], truncation=True, padding="max_length", return_tensors="pt").input_ids
    examples['labels']  = tokenizer(examples['answer'],  truncation=True, padding="max_length", return_tensors="pt").input_ids
    
    print("Input IDs:")
    print("Data Type of input:", examples['inputs'].dtype)
    print("Data Type of output:", examples['labels'].dtype)
    
    print("Shape IDs:")
    print("Data Type of input:", examples['inputs'].shape)
    print("Data Type of output:", examples['labels'].shape)
    
    
    return examples

In [280]:
tokenized_datasets = df.map(preprocess_function, batched=True)

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Input IDs:
Data Type of input: torch.int64
Data Type of output: torch.int64
Shape IDs:
Data Type of input: torch.Size([800, 512])
Data Type of output: torch.Size([800, 512])


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Input IDs:
Data Type of input: torch.int64
Data Type of output: torch.int64
Shape IDs:
Data Type of input: torch.Size([200, 512])
Data Type of output: torch.Size([200, 512])


In [281]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'inputs', 'labels'],
        num_rows: 800
    })
    test: Dataset({
        features: ['question', 'answer', 'inputs', 'labels'],
        num_rows: 200
    })
})

In [282]:
tokenized_datasets = tokenized_datasets.remove_columns(['question','answer'])

In [283]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['inputs', 'labels'],
        num_rows: 800
    })
    test: Dataset({
        features: ['inputs', 'labels'],
        num_rows: 200
    })
})

In [290]:
# small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(800))
# small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(200))
# print(small_train_dataset.shape)
# print(small_eval_dataset.shape)


In [291]:
output_dir = 'checkpoints'

training_args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            learning_rate=3e-5,
            evaluation_strategy="steps",
            # logging_strategy='steps',
            logging_steps=100,
            # save_strategy="epoch",
            # save_steps=10,
            num_train_epochs=10,
)

In [292]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
)

In [293]:
trainer.train()

  0%|          | 0/8000 [00:00<?, ?it/s]

IndexError: Invalid key: 742 is out of bounds for size 0

In [270]:
# # Tokenize and encode input and output sequences
# def preprocess_function(examples):
#     inputs = tokenizer(examples['question'], truncation=True, padding="max_length", return_tensors="pt").input_ids
#     outputs = tokenizer(examples['answer'],  truncation=True, padding="max_length", return_tensors="pt").input_ids
    
#     print("Input IDs:")
#     print("Data Type:", inputs.input_ids.dtype)
#     print("Data Type of :", inputs.input_ids.shape)
    
#     print("Labels:")
#     print("Data Type:", outputs.input_ids.dtype)
#     print("Shape:", outputs.input_ids.shape)
    
#     # Update examples with inputs and outputs
#     examples["input_ids"] = inputs.input_ids
#     # examples["attention_mask"] = inputs.attention_mask
#     examples["labels"] = outputs.input_ids
#     # examples["start_positions"] = outputs.input_ids
#     # examples["end_positions"] = outputs.input_ids
    
#     # examples["labels"] = torch.tensor(examples["labels"])
#     return examples

In [271]:
# train_dataset = df['train'].map(preprocess_function, batched=True).select(range(800))
# test_dataset = df['test'].map(preprocess_function, batched=True).select(range(200))

In [272]:
train_dataset = train_dataset.remove_columns(['question','answer'])
# test_dataset = test_dataset.remove_columns(['question','answer'])

In [273]:
# Define training arguments
output_dir = 'checkpoints'
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=3e-5,
    evaluation_strategy="epoch",
    logging_strategy='steps',
    logging_steps=10,
    save_strategy="epoch",
    save_steps=10,
    num_train_epochs=10,
)


In [252]:
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        outputs = model(**inputs)
        start_logits = outputs.start_logits
        end_logits = outputs.end_logits
        
        start_positions = inputs["start_positions"]
        end_positions = inputs["end_positions"]
        
        # Print unique values
        print("Unique values:")
        print("start_positions:", torch.unique(start_positions))
        print("end_positions:", torch.unique(end_positions))
        
        # Print shapes
        print("Shapes:")
        print("start_logits:", start_logits.shape)
        print("end_logits:", end_logits.shape)
        print("start_positions:", start_positions.shape)
        print("end_positions:", end_positions.shape)
        
        # Ensure start and end positions are flattened to 1D tensors
        start_positions = start_positions.flatten()
        end_positions = end_positions.flatten()
        
        # Calculate loss
        loss_fct = torch.nn.CrossEntropyLoss(ignore_index=-100)
        start_loss = loss_fct(start_logits.view(-1, start_logits.size(-1)), start_positions)
        end_loss = loss_fct(end_logits.view(-1, end_logits.size(-1)), end_positions)
        total_loss = (start_loss + end_loss) / 2
        
        if return_outputs:
            return total_loss, outputs
        return total_loss



In [274]:
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)


In [275]:
# Train the model
trainer.train()

  0%|          | 0/8000 [00:00<?, ?it/s]

IndexError: Invalid key: 742 is out of bounds for size 0